# Published NFPT fixed baseline and live NFGS–NF2 comparison

Published scores retain their original protocol and values. New Pareto/timing results use common canonical inputs and FD2/GS. Original-publication equivalence remains an explicit verification gate. Local PT results are supplementary, and never fill missing published CCI/runtime.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from nfgs.paths import project_root
ROOT = project_root()
pointer = ROOT / '.local/controlled/active_root'
RUN = Path(pointer.read_text().strip()) if pointer.exists() else ROOT / 'results/controlled/live_20261014'
def read_json(path, default):
    return json.loads(path.read_text()) if path.exists() else default
state = read_json(RUN / 'campaign_state.json', {'jobs': [], 'status': 'not run locally'})
policy = read_json(RUN / 'comparison_policy.json', {})
jobs = pd.DataFrame(state['jobs'])
display(pd.Series({'status': state['status'], 'policy': policy.get('policy_id'), 'active_jobs': len(jobs),
                  'completed': int((jobs.status == 'completed').sum()) if len(jobs) else 0,
                  'original_NFPT_protocol_verified': policy.get('published_protocol_match_verified', False)}))


## Published scores — fixed values, original evaluation protocol

Missing CCI/runtime are retained. The table is separate from common-protocol new measurements.


In [ ]:
published_path = RUN / 'published_baseline.csv'
if not published_path.exists():
    published_path = ROOT / 'reference_results/published_methods.csv'
published = pd.read_csv(published_path, dtype=str)
display(published)


## Executed NFGS–NF2 points — partial groups remain visible

Only completed runs are plotted. Individual NF2 seeds are shown; a partial weight group is not interpreted as a complete five-seed summary.


In [ ]:
# Read completed run JSON directly so the notebook stays current between report builds.
live_rows = []
for job in state['jobs']:
    if job['status'] != 'completed' or job['method'] not in ['NFGS','NF2']:
        continue
    result = read_json(RUN / 'runs' / job['id'] / 'result.json', {})
    record = read_json(RUN / 'runs' / job['id'] / 'metrics.json', {})
    if not result or not record:
        continue
    row = {'id': job['id'], 'status': job['status'], 'stage': job['stage'],
           'case': job['case_spec']['id'], 'method': job['method'], **job['settings']}
    row.update({key: value for key, value in record['metrics'].items() if isinstance(value, (int,float,str)) or value is None})
    live_rows.append(row)
measurements = pd.DataFrame(live_rows)
if len(measurements):
    main = measurements[(measurements.status == 'completed') & (measurements.stage == 'primary') & measurements.method.isin(['NFGS','NF2'])]
    display(main[['case','method','rho','boundary_weight_multiplier','seed','C_vec','sigma_J','E_rel_Bgt']])
    for case, block in main.groupby('case'):
        fig, ax = plt.subplots(figsize=(6,4))
        for method, part in block.groupby('method'):
            ax.scatter(part.E_rel_Bgt, part.sigma_J, label=method, alpha=.75)
        ax.set(xlabel='CCI relative error', ylabel=r'$\sigma_J$', title=case+' — common FD2/GS')
        ax.legend(); ax.grid(alpha=.2); plt.show()
    nf2 = main[main.method == 'NF2']
    if len(nf2):
        counts = nf2.groupby(['case','boundary_weight_multiplier']).seed.nunique().rename('completed_seeds').reset_index()
        counts['complete_five_seed_group'] = counts.completed_seeds == 5
        display(counts)
else:
    print('Run the report builder after local experiments exist.')


## Timing and preserved evidence

Timing groups keep numerical settings separate. Paused wall time is excluded with the original values retained in provenance. Published NFPT has no current-hardware solution-time measurement.


In [ ]:
display(pd.DataFrame(read_json(RUN / 'report/timing_mean_sd.json', [])))
excluded = read_json(RUN / 'policy_revisions/published_nfpt_fixed_v2/excluded_jobs.json', [])
display(pd.Series({'preserved_local_PT_jobs': len(excluded), 'published_NFPT_used_as_runtime_baseline': False}))
display(pd.Series(policy.get('completed_external_evidence', {})))
